# Step 3: BRCA cohort

BRCA patients from the survival table, joined with clinical data from cBioPortal: PAM50 subtype, ER/PR/HER2 status, stage, and age.

cBioPortal doesn't have all of these in one study:
- PAM50 (`SUBTYPE`) is in **TCGA PanCancer Atlas** (`brca_tcga_pan_can_atlas_2018`)
- ER/PR/HER2 are in **TCGA Firehose Legacy** (`brca_tcga`)

Age and stage are already in the survival table, so those come from there. cBioPortal's copies are used as a cross-check.

Output: `brca_cohort_clinical.csv`, one row per patient.

In [1]:
import pandas as pd
import numpy as np
import requests

CBIO_API = "https://www.cbioportal.org/api"

## 1. BRCA patients from the survival table

In [2]:
survival = pd.read_csv("Survival_SupplementalTable_S1_20171025_xena_sp", sep="\t")
brca = survival[survival["cancer type abbreviation"] == "BRCA"].copy()
print("BRCA rows:", len(brca), "| patients:", brca["_PATIENT"].nunique())

# The table has one row per sample. Some patients also have a normal (-11) or metastatic (-06)
# sample, so keep the primary tumor (-01) row to get one row per patient.
brca["sample_type"] = brca["sample"].str[13:15]
print("Sample types:", brca["sample_type"].value_counts().to_dict())
brca = brca[brca["sample_type"] == "01"]
assert brca["_PATIENT"].is_unique

cohort = brca.rename(columns={
    "_PATIENT": "patient_id",
    "age_at_initial_pathologic_diagnosis": "age",
    "ajcc_pathologic_tumor_stage": "stage",
})[["patient_id", "sample", "age", "gender", "stage", "OS", "OS.time", "PFI", "PFI.time"]]
print("Cohort:", len(cohort), "patients")
cohort.head()

BRCA rows: 1236 | patients: 1097
Sample types: {'01': 1097, '11': 132, '06': 7}
Cohort: 1097 patients


,patient_id,sample,age,gender,stage,OS,OS.time,PFI,PFI.time
528,TCGA-3C-AAAU,TCGA-3C-AAAU-01,55.0,FEMALE,Stage X,0.0,4047.0,1.0,1808.0
529,TCGA-3C-AALI,TCGA-3C-AALI-01,50.0,FEMALE,Stage IIB,0.0,4005.0,0.0,4005.0
530,TCGA-3C-AALJ,TCGA-3C-AALJ-01,62.0,FEMALE,Stage IIB,0.0,1474.0,0.0,1474.0
531,TCGA-3C-AALK,TCGA-3C-AALK-01,52.0,FEMALE,Stage IA,0.0,1448.0,0.0,1448.0
532,TCGA-4H-AAAK,TCGA-4H-AAAK-01,50.0,FEMALE,Stage IIIA,0.0,348.0,0.0,348.0


## 2. Pull clinical data from cBioPortal

In [3]:
def cbio_patient_data(study_id, attributes):
    """Patient-level clinical attributes for a cBioPortal study, one row per patient."""
    r = requests.get(
        f"{CBIO_API}/studies/{study_id}/clinical-data",
        params={"clinicalDataType": "PATIENT", "projection": "SUMMARY"},
        timeout=120,
    )
    r.raise_for_status()
    long = pd.DataFrame(r.json())
    long = long[long["clinicalAttributeId"].isin(attributes)]
    wide = long.pivot(index="patientId", columns="clinicalAttributeId", values="value")
    return wide.reindex(columns=attributes).rename_axis("patient_id").reset_index()

pancan = cbio_patient_data("brca_tcga_pan_can_atlas_2018", ["SUBTYPE", "AGE", "AJCC_PATHOLOGIC_TUMOR_STAGE"])
legacy = cbio_patient_data("brca_tcga", ["ER_STATUS_BY_IHC", "PR_STATUS_BY_IHC", "IHC_HER2", "HER2_FISH_STATUS"])
print("PanCancer Atlas:", len(pancan), "patients | Firehose Legacy:", len(legacy), "patients")

PanCancer Atlas: 1084 patients | Firehose Legacy: 1049 patients


## 3. Clean up

- PAM50: drop the `BRCA_` prefix (`BRCA_LumA` becomes `LumA`)
- ER/PR: `Indeterminate` becomes missing
- HER2: IHC `Positive`/`Negative` is used as is. `Equivocal` (IHC 2+) is resolved by FISH, which is the standard clinical rule. Raw IHC and FISH columns are kept.
- Stage: also collapsed to `I` to `IV` (`stage_group`), since substages like IIA/IIB get small

In [4]:
clin = pancan.merge(legacy, on="patient_id", how="outer")

clin["pam50"] = clin["SUBTYPE"].str.replace("BRCA_", "", regex=False)

def pos_neg(s):
    return s.where(s.isin(["Positive", "Negative"]))

clin["er_status"] = pos_neg(clin["ER_STATUS_BY_IHC"])
clin["pr_status"] = pos_neg(clin["PR_STATUS_BY_IHC"])

her2_ihc = clin["IHC_HER2"]
her2_fish = pos_neg(clin["HER2_FISH_STATUS"])
clin["her2_status"] = np.where(her2_ihc.isin(["Positive", "Negative"]), her2_ihc,
                      np.where(her2_ihc == "Equivocal", her2_fish, None))
clin["her2_status"] = clin["her2_status"].where(clin["her2_status"].isin(["Positive", "Negative"]))

clin = clin.rename(columns={
    "AGE": "age_cbio",
    "AJCC_PATHOLOGIC_TUMOR_STAGE": "stage_cbio",
    "IHC_HER2": "her2_ihc_raw",
    "HER2_FISH_STATUS": "her2_fish_raw",
})[["patient_id", "pam50", "er_status", "pr_status", "her2_status", "her2_ihc_raw", "her2_fish_raw", "age_cbio", "stage_cbio"]]

print("HER2 equivocal by IHC:", (her2_ihc == "Equivocal").sum(),
      "| resolved by FISH:", ((her2_ihc == "Equivocal") & her2_fish.notna()).sum())

HER2 equivocal by IHC: 179 | resolved by FISH: 158


## 4. Join on patient ID

In [5]:
brca_cohort = cohort.merge(clin, on="patient_id", how="left")

def stage_group(stage):
    if pd.isna(stage) or not str(stage).upper().startswith("STAGE "):
        return np.nan
    roman = str(stage).upper().replace("STAGE ", "").rstrip("ABC")
    return roman if roman in {"I", "II", "III", "IV"} else np.nan

brca_cohort["stage_group"] = brca_cohort["stage"].map(stage_group)

matched = brca_cohort["patient_id"].isin(clin["patient_id"])
print(f"Matched to cBioPortal: {matched.sum()} / {len(brca_cohort)} patients")
print("Not in cBioPortal:", brca_cohort.loc[~matched, "patient_id"].tolist())

Matched to cBioPortal: 1097 / 1097 patients
Not in cBioPortal: []


Missing values per column (out of the full cohort):

In [6]:
cols = ["age", "stage_group", "pam50", "er_status", "pr_status", "her2_status", "OS", "OS.time", "PFI", "PFI.time"]
pd.DataFrame({
    "missing": brca_cohort[cols].isna().sum(),
    "percent": (brca_cohort[cols].isna().mean() * 100).round(1),
})

,missing,percent
age,1,0.1
stage_group,24,2.2
pam50,116,10.6
er_status,51,4.6
pr_status,54,4.9
her2_status,211,19.2
OS,0,0.0
OS.time,1,0.1
PFI,0,0.0
PFI.time,1,0.1


Value counts:

In [7]:
for col in ["pam50", "er_status", "pr_status", "her2_status", "stage_group", "gender"]:
    print(col, brca_cohort[col].value_counts(dropna=False).to_dict())

pam50 {'LumA': 499, 'LumB': 197, 'Basal': 171, nan: 116, 'Her2': 78, 'Normal': 36}
er_status {'Positive': 808, 'Negative': 238, nan: 51}
pr_status {'Positive': 699, 'Negative': 344, nan: 54}
her2_status {'Negative': 703, nan: 211, 'Positive': 183}
stage_group {'II': 621, 'III': 249, 'I': 183, nan: 24, 'IV': 20}
gender {'FEMALE': 1085, 'MALE': 12}


## 5. Sanity checks

- Age and stage from the survival table vs cBioPortal should mostly agree
- PAM50 vs receptor status should look right: LumA/LumB mostly ER+, Basal mostly ER-, Her2 enriched for HER2+

In [8]:
both = brca_cohort.dropna(subset=["age", "age_cbio"])
print("Age matches cBioPortal:", (both["age"].astype(int) == both["age_cbio"].astype(int)).mean().round(3), f"(n={len(both)})")

both = brca_cohort.dropna(subset=["stage", "stage_cbio"])
print("Stage matches cBioPortal:", (both["stage"].str.upper() == both["stage_cbio"].str.upper()).mean().round(3), f"(n={len(both)})")

print()
print(pd.crosstab(brca_cohort["pam50"], brca_cohort["er_status"], margins=True))
print()
print(pd.crosstab(brca_cohort["pam50"], brca_cohort["her2_status"], margins=True))

Age matches cBioPortal: 1.0 (n=1082)
Stage matches cBioPortal: 0.994 (n=1078)

er_status  Negative  Positive  All
pam50                             
Basal           144        20  164
Her2             46        26   72
LumA             11       469  480
LumB              3       182  185
Normal           13        20   33
All             217       717  934

her2_status  Negative  Positive  All
pam50                               
Basal             126        12  138
Her2               15        49   64
LumA              336        61  397
LumB              123        36  159
Normal             23         3   26
All               623       161  784


## 6. Expression data overlap

Which cohort patients have a primary tumor sample in the expression data. Needs `vae_embeddings_latent32.csv` from the Echo notebook (same samples as `adata_tcga_preprocessed.h5ad`).

In [9]:
import os

if os.path.exists("vae_embeddings_latent32.csv"):
    expr_samples = pd.read_csv("vae_embeddings_latent32.csv", usecols=[0]).iloc[:, 0]
    expr_tumor_patients = set(expr_samples[expr_samples.str[13:15] == "01"].str[:12])
    brca_cohort["has_expression"] = brca_cohort["patient_id"].isin(expr_tumor_patients)
    print("Cohort patients with a primary tumor expression sample:", brca_cohort["has_expression"].sum(), "/", len(brca_cohort))
else:
    print("vae_embeddings_latent32.csv not found, skipping")

Cohort patients with a primary tumor expression sample: 1095 / 1097


## 7. Save

Note: 12 male patients are included (`gender` column), filter them out if needed.

In [10]:
brca_cohort.to_csv("brca_cohort_clinical.csv", index=False)
print(brca_cohort.shape)
brca_cohort.head()

(1097, 19)


,patient_id,sample,age,gender,stage,OS,OS.time,PFI,PFI.time,pam50,er_status,pr_status,her2_status,her2_ihc_raw,her2_fish_raw,age_cbio,stage_cbio,stage_group,has_expression
0,TCGA-3C-AAAU,TCGA-3C-AAAU-01,55.0,FEMALE,Stage X,0.0,4047.0,1.0,1808.0,LumA,Positive,Positive,Negative,Negative,[Not Evaluated],55,STAGE X,NaN,True
1,TCGA-3C-AALI,TCGA-3C-AALI-01,50.0,FEMALE,Stage IIB,0.0,4005.0,0.0,4005.0,Her2,Positive,Positive,Positive,Positive,NaN,50,STAGE IIB,II,True
2,TCGA-3C-AALJ,TCGA-3C-AALJ-01,62.0,FEMALE,Stage IIB,0.0,1474.0,0.0,1474.0,LumB,Positive,Positive,NaN,Indeterminate,NaN,62,STAGE IIB,II,True
3,TCGA-3C-AALK,TCGA-3C-AALK-01,52.0,FEMALE,Stage IA,0.0,1448.0,0.0,1448.0,LumA,Positive,Positive,Positive,Positive,NaN,52,STAGE IA,I,True
4,TCGA-4H-AAAK,TCGA-4H-AAAK-01,50.0,FEMALE,Stage IIIA,0.0,348.0,0.0,348.0,LumA,Positive,Positive,NaN,Equivocal,NaN,50,STAGE IIIA,III,True
